In [4]:
import pandas as pd
from google.colab import files
import io

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  # Assuming the uploaded file is a CSV, read it into a pandas DataFrame
  df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
  break # Assuming only one file needs to be processed or the first one is sufficient

Saving train.csv to train.csv
User uploaded file "train.csv" with length 39984826 bytes


In [5]:
# Display the first 5 rows of the DataFrame
display(df.head())

,instruction,output,input
0,Give three tips for staying healthy.,1. Eat a balanced and nutritious diet: Make su...,NaN
1,What are the three primary colors?,"The three primary colors are red, blue, and ye...",NaN
2,Describe the structure of an atom.,An atom is the basic building block of all mat...,NaN
3,How can we reduce air pollution?,There are several ways to reduce air pollution...,NaN
4,Pretend you are a project manager of a constru...,I had to make a difficult decision when I was ...,NaN


## 1. Install Necessary Libraries

First, we need to install `transformers` for the model and tokenizer, `peft` for LoRA, `accelerate` for optimized training, `bitsandbytes` for 8-bit quantization (if needed, which can save memory), and `trl` (Transformer Reinforcement Learning) for `SFTTrainer` which simplifies supervised fine-tuning.

In [18]:
!pip install -q -U transformers peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.2 MB/s eta 0:00:00


In [19]:
# Upgrade torchao to a compatible version
!pip install -q -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.9 MB/s eta 0:00:00


After installing `torchao`, please re-run the cells from `f7ac42d3` (model loading) to `87a5fdec` (SFTTrainer initialization) to ensure all dependencies are correctly loaded with the new `torchao` version and the updated optimizer.

## 2. Load Model and Tokenizer

Next, we'll load the Qwen2.5-0.5B model and its corresponding tokenizer. We'll load the model in 4-bit precision to save memory, which is common for LoRA fine-tuning. We'll also set up the `device_map` to automatically handle model placement (e.g., on GPU if available).

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer # Removed BitsAndBytesConfig as it's not needed for non-quantized loading

model_id = "Qwen/Qwen2-0.5B-Instruct"

# Load model without quantization (as requested)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False # Disable cache for training

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

We need to add a padding token to the tokenizer if it doesn't have one, and resize the model's token embeddings.

In [12]:
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Or another suitable token like <|endoftext|>
    model.resize_token_embeddings(len(tokenizer))

## 3. Prepare the Dataset for LoRA Fine-tuning

The `train.csv` dataset contains `instruction`, `output`, and `input` columns. For fine-tuning a language model, we need to format these into a conversational or instruction-following format. We'll create a `text` column that combines `instruction` and `input` to form the prompt, and then append the `output`.

In [16]:
from datasets import Dataset

# The df variable is already available from the previous upload cell
# df = pd.read_csv('train.csv') # Uncomment if you need to load it again

# Function to format the dataset
def formatting_function(example):
    text = f"### Instruction:\n{example['instruction']}\n"
    if example['input']:
        text += f"### Input:\n{example['input']}\n"
    text += f"### Output:\n{example['output']}\n"
    return {"text": text}

# Convert pandas DataFrame to Hugging Face Dataset, using only the first 5000 samples
dataset = Dataset.from_pandas(df.head(5000))

# Apply the formatting function
formatted_dataset = dataset.map(formatting_function)

print(formatted_dataset[0]['text']) # Display one example to check formatting

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

### Instruction:
Give three tips for staying healthy.
### Output:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.



### Fix for `pyarrow` and `datasets` incompatibility

The `ValueError: pyarrow.lib.IpcReadOptions size changed...` typically indicates a version mismatch between `pyarrow` and `datasets`. The following cell will uninstall and reinstall these libraries to ensure they are compatible. Please run this cell, and then **re-run cell `4c3e1ade`**.

In [20]:
# Uninstall and reinstall pyarrow and datasets to resolve version conflicts
!pip uninstall -y pyarrow datasets
!pip install -q -U pyarrow datasets

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0


## 4. Configure LoRA

We'll set up the LoRA configuration using `LoraConfig`. This defines parameters like `r` (LoRA attention dimension), `lora_alpha` (scaling factor), `target_modules` (which layers to apply LoRA to), and `task_type`.

In [14]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target all attention and feed-forward layers for Qwen2
    bias="none", # Only training bias will make the training unstable
    lora_dropout=0.05, # Dropout probability for LoRA layers
    task_type="CAUSAL_LM", # Causal language modeling task
)

## 5. Set up Training Arguments and Trainer

We'll use `TrainingArguments` from `transformers` to define our training parameters and `SFTTrainer` from `trl` for supervised fine-tuning. `SFTTrainer` handles data collation and formatting automatically when given a `formatting_function` or a pre-formatted dataset.

In [20]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results", # Directory to save the model checkpoints
    num_train_epochs=1, # Number of training epochs
    per_device_train_batch_size=4, # Batch size per GPU/CPU
    gradient_accumulation_steps=1, # Number of updates steps to accumulate before performing a backward/update pass
    gradient_checkpointing=True, # Enable gradient checkpointing to save memory
    optim="paged_adamw_8bit", # Optimizer for 8-bit quantization
    learning_rate=2e-4, # Learning rate
    fp16=True, # Use half-precision floats (recommended for faster training)
    logging_steps=100, # Log training metrics every 100 steps
    save_steps=500, # Save checkpoint every 500 steps
    report_to="none", # Don't report to any external service
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=lora_config,
    args=training_args
)

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

## 6. Train the Model

Now, let's start the training process. This might take some time depending on your dataset size and available hardware (GPU is highly recommended for training).

In [21]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
100,1.526970
200,1.480093
300,1.444951
400,1.431385
500,1.443979
600,1.498903
700,1.478041
800,1.443087
900,1.426766
1000,1.480665


TrainOutput(global_step=1250, training_loss=1.454294287109375, metrics={'train_runtime': 819.7399, 'train_samples_per_second': 6.099, 'train_steps_per_second': 1.525, 'total_flos': 3203460967649280.0, 'train_loss': 1.454294287109375, 'entropy': 1.4255126905441284, 'num_tokens': 792731.0, 'mean_token_accuracy': 0.6485478198528289, 'epoch': 1.0})

## 7. Save the Fine-tuned Model

After training, we'll save the LoRA adapters. Optionally, we can also merge the adapters with the base model to create a single, portable model. Merging usually requires more RAM, so if you're memory-constrained, just saving the adapters is sufficient.

In [22]:
# Save the LoRA adapters
output_dir = "./qwen2_0_5b_lora_fine_tuned"
trainer.save_model(output_dir)
print(f"LoRA adapters saved to {output_dir}")

# Optional: Merge LoRA adapters with the base model and save (requires more RAM)
# You might need to reload the base model on a CPU to merge if GPU memory is limited after training.
# For merging, it's often easier to do it after loading the adapters for inference.
# For now, let's just save the adapters.

LoRA adapters saved to ./qwen2_0_5b_lora_fine_tuned


## 8. Inference and Run on CPU

To run inference on a CPU, we need to load the base model and then load the saved LoRA adapters. Ensure `device_map='cpu'` or explicitly move the model to CPU. This section assumes you have sufficient RAM on your CPU to load the `0.5B` model.

In [23]:
from peft import PeftModel, PeftConfig

# Clear GPU memory if any (important if you trained on GPU and now want to use CPU)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load the base model directly on CPU without quantization for inference
# If you saved the merged model, load that instead of the base + adapters
print("Loading base model on CPU...")
base_model_cpu = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu", # Explicitly set to CPU
    torch_dtype=torch.float16, # Use float16 for smaller memory footprint on CPU if desired
    trust_remote_code=True
)

# Load the tokenizer again
tokenizer_cpu = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer_cpu.pad_token is None:
    tokenizer_cpu.add_special_tokens({'pad_token': '[PAD]'}) # Or another suitable token like <|endoftext|>

# Load the LoRA adapters onto the base model
print("Loading LoRA adapters...")
model_for_inference = PeftModel.from_pretrained(base_model_cpu, output_dir)

# Optionally, merge the adapters with the base model for a single model
print("Merging adapters with base model...")
model_for_inference = model_for_inference.merge_and_unload()
model_for_inference.eval()

print("Model ready for inference on CPU.")

Loading base model on CPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading LoRA adapters...
Merging adapters with base model...
Model ready for inference on CPU.


Now, let's test inference with the fine-tuned model on the CPU.

In [24]:
prompt = "### Instruction:\nWhat is the capital of France?\n### Input:\n\n### Output:\n"

# Tokenize the prompt
inputs = tokenizer_cpu(prompt, return_tensors="pt").to("cpu") # Ensure inputs are on CPU

# Generate a response
with torch.no_grad():
    outputs = model_for_inference.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer_cpu.pad_token_id
    )

response = tokenizer_cpu.decode(outputs[0], skip_special_tokens=True)
print("Generated Response:")
print(response)

# Example with input
prompt_with_input = "### Instruction:\nSummarize the following text:\n### Input:\nParis is the capital and most populous city of France, with an estimated population of 2,140,526 residents as of 2019.\n### Output:\n"
inputs_with_input = tokenizer_cpu(prompt_with_input, return_tensors="pt").to("cpu")

with torch.no_grad():
    outputs_with_input = model_for_inference.generate(
        **inputs_with_input,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer_cpu.pad_token_id
    )

response_with_input = tokenizer_cpu.decode(outputs_with_input[0], skip_special_tokens=True)
print("\nGenerated Response (with input):")
print(response_with_input)


Generated Response:
### Instruction:
What is the capital of France?
### Input:

### Output:
The capital city of France is Paris.


Generated Response (with input):
### Instruction:
Summarize the following text:
### Input:
Paris is the capital and most populous city of France, with an estimated population of 2,140,526 residents as of 2019.
### Output:
The text describes Paris as the capital and most populous city in France. As of 2019, it had a population of approximately 2,140,526 residents.



In [25]:
import os
from google.colab import files

output_dir = "./qwen2_0_5b_lora_fine_tuned"

# List all files in the output directory
print(f"Files in '{output_dir}':")
for filename in os.listdir(output_dir):
    print(filename)
    # Download each file
    files.download(os.path.join(output_dir, filename))

print("Download process complete.")

Files in './qwen2_0_5b_lora_fine_tuned':
adapter_config.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

tokenizer.json


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

tokenizer_config.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download process complete.
